<a href="https://colab.research.google.com/github/zeynepoztunc/aml-procedural-mistake-detection/blob/zeynep-september/extension_step1_actionformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Extension Step 1: Recipe Step Localization with ActionFormer

Implements **Substep 1 only** of the "From Mistake Detection to Task Verification" extension:

1. Use a pre-trained step localization approach to get `(start, end)` timestamps per detected step.
2. For each detected step, mean-pool the EgoVLP features falling inside that interval.
3. Produce, per recording, a sequence of step-level embeddings.

Uses the **official released** CaptainCook4D ActionFormer predictions (no retraining, no grid
search, no KMeans/change-point detection). Substep 2-4 (task-graph matching / classification) is
out of scope for this notebook.

<a href="https://colab.research.google.com/github/zeynepoztunc/aml-procedural-mistake-detection/blob/zeynep-september/notebooks/extension_step1_actionformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Setup (Colab)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

REPO_URL = "https://github.com/zeynepoztunc/aml-procedural-mistake-detection.git"
BRANCH = "zeynep-september"
CODE_DIR = "/content/code"

# Colab's "Restart session" resets the Python process but NOT the disk -- if /content/code
# already exists from an earlier run, a plain `git clone` fails (target not empty) and, since
# `!` shell errors don't raise in a notebook, execution would silently continue against a
# stale old clone. Pull instead of no-op'ing when the directory is already there.
if os.path.exists(CODE_DIR):
    print(f"{CODE_DIR} already exists -- pulling latest instead of re-cloning.")
    !cd {CODE_DIR} && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !git clone --recursive --branch {BRANCH} {REPO_URL} {CODE_DIR}

Cloning into '/content/code'...
remote: Enumerating objects: 926, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 926 (delta 111), reused 94 (delta 91), pack-reused 786 (from 2)
Receiving objects: 100% (926/926), 96.56 MiB | 18.15 MiB/s, done.
Resolving deltas: 100% (501/501), done.
Submodule 'annotations' (https://github.com/CaptainCook4D/annotations) registered for path 'annotations'
Cloning into '/content/code/annotations'...
remote: Enumerating objects: 152, done.        
remote: Counting objects: 100% (152/152), done.        
remote: Compressing objects: 100% (98/98), done.        
remote: Total 152 (delta 75), reused 108 (delta 46), pack-reused 0 (from 0)        
Receiving objects: 100% (152/152), 793.14 KiB | 17.24 MiB/s, done.
Resolving deltas: 100% (75/75), done.
Submodule path 'annotations': checked out '0e9a108be2cbcbcbd592e7418c0ab9c16232d27a'


In [3]:
%cd /content/code

/content/code


## 2. Locate existing EgoVLP features

Reads directly from wherever your EgoVLP `.npz` files already live on Drive -- no repo dataloader
code is used or modified here, this is pure numpy/csv processing on the feature arrays. Handles
either filename convention seen in this project so far (`<recording_id>_360p.mp4_1s_1s.npz` from
this repo's own extraction notebook, or `<recording_id>_360p_224.npz`) and either array key
(`arr_0` or `video_features`), without renaming or copying anything.

In [4]:
import numpy as np

DRIVE_BASE = "/content/drive/MyDrive/AML_Project"

EGOVLP_CANDIDATE_DIRS = [
    os.path.join(DRIVE_BASE, "features/egovlp"),
    os.path.join(DRIVE_BASE, "features", "egovlp"),
    "/content/code/data/video/egovlp",
]
KNOWN_SUFFIXES = ["_360p_224.npz", "_360p.mp4_1s_1s.npz", ".npz"]

EGOVLP_DIR = next((d for d in EGOVLP_CANDIDATE_DIRS if os.path.isdir(d)), None)
assert EGOVLP_DIR is not None, (
    f"No EgoVLP feature directory found. Checked: {EGOVLP_CANDIDATE_DIRS}"
)
print(f"Using EgoVLP feature directory: {EGOVLP_DIR}")

# recording_id -> filename
_egovlp_files = {}
for fname in os.listdir(EGOVLP_DIR):
    if not fname.endswith(".npz"):
        continue
    recording_id = fname
    for suffix in KNOWN_SUFFIXES:
        if fname.endswith(suffix):
            recording_id = fname[: -len(suffix)]
            break
    _egovlp_files[recording_id] = fname

print(f"Found EgoVLP features for {len(_egovlp_files)} recordings")
print(f"Sample recording_ids: {sorted(_egovlp_files)[:5]}")


def load_egovlp_features(recording_id):
    """Load the (num_seconds, 256) EgoVLP feature array for a recording, or None."""
    fname = _egovlp_files.get(recording_id)
    if fname is None:
        return None
    data = np.load(os.path.join(EGOVLP_DIR, fname))
    if "arr_0" in data:
        return data["arr_0"]
    if "video_features" in data:
        return data["video_features"]
    return data[list(data.keys())[0]]


# Peek at one file to confirm the feature dimension
_sample_id = sorted(_egovlp_files)[0]
_sample_feat = load_egovlp_features(_sample_id)
print(f"Sample features ({_sample_id}): shape={_sample_feat.shape}, dtype={_sample_feat.dtype}")

Using EgoVLP feature directory: /content/drive/MyDrive/AML_Project/features/egovlp
Found EgoVLP features for 384 recordings
Sample recording_ids: ['10_16', '10_18', '10_24', '10_26', '10_31']
Sample features (10_16): shape=(973, 256), dtype=float16


## 3. Feature time base

EgoVLP features here are 1 vector per second (`SEGMENT_LENGTH=1` in `egovlp_feature_extraction.ipynb`),
so feature row *i* corresponds to second *i* of the recording: `FEATURE_FPS = 1.0`. This project has
previously broken silently when features and annotation timestamps were on different time bases
(numpy returns an *empty* slice for an out-of-range index rather than raising) -- so this stays an
explicit, named constant rather than an inline literal.

In [5]:
FEATURE_FPS = 1.0
print(f"Using FEATURE_FPS = {FEATURE_FPS} (1 feature vector per second)")

Using FEATURE_FPS = 1.0 (1 feature vector per second)


## 4. Download the official pre-trained ActionFormer predictions

The released CaptainCook4D ActionFormer output (not a model we run -- its published *predictions*).
No published weights exist for this model and it was trained on different (Omnivore, 4s) features,
so it can't be re-run on our EgoVLP features anyway; the released `(start, end)` segments are exactly
what Substep 1 asks this stage to produce, and they're genuine held-out predictions (test split of
the recordings division, 119 of 384 recordings, all 24 recipes represented).

In [6]:
import csv
import io
import urllib.request
import collections

PRETRAINED_AF_URL = (
    "https://raw.githubusercontent.com/CaptainCook4D/multi_step_localization/"
    "main/model_outputs/pred_segments_error_dataset.csv"
)

OUTPUT_DIR = os.path.join(DRIVE_BASE, "extension_data", "actionformer_egovlp")
os.makedirs(OUTPUT_DIR, exist_ok=True)
AF_CACHE_PATH = os.path.join(OUTPUT_DIR, "pred_segments_error_dataset.csv")

if os.path.exists(AF_CACHE_PATH):
    with open(AF_CACHE_PATH, "r") as f:
        _af_csv_text = f.read()
    print(f"Loaded cached predictions: {AF_CACHE_PATH}")
else:
    print(f"Downloading official predictions...\n  {PRETRAINED_AF_URL}")
    with urllib.request.urlopen(PRETRAINED_AF_URL) as resp:
        _af_csv_text = resp.read().decode("utf-8")
    with open(AF_CACHE_PATH, "w") as f:
        f.write(_af_csv_text)
    print(f"  cached -> {AF_CACHE_PATH}")

  https://raw.githubusercontent.com/CaptainCook4D/multi_step_localization/main/model_outputs/pred_segments_error_dataset.csv
  cached -> /content/drive/MyDrive/AML_Project/extension_data/actionformer_egovlp/pred_segments_error_dataset.csv


## 5. Inspect the file before processing

Confirm the columns and timestamp units before trusting them.

In [7]:
_af_reader = list(csv.DictReader(io.StringIO(_af_csv_text)))
print(f"Total rows: {len(_af_reader)}")
print(f"Columns: {list(_af_reader[0].keys())}")
print("\nFirst 5 rows:")
for row in _af_reader[:5]:
    print(f"  {row}")

_types = collections.Counter(row["type"] for row in _af_reader)
print(f"\n'type' value counts: {dict(_types)}")

_max_t = max(float(row["t-end"]) for row in _af_reader if row["t-end"])
print(f"Max t-end across the file: {_max_t:.1f} (seconds -- consistent with a recording-length scale, not frames)")

Total rows: 3146
Columns: ['video-id', 't-start', 't-end', 'label', 'type', 'score']

First 5 rows:
  {'video-id': '1_20', 't-start': '0.0', 't-end': '101.871', 'label': '12', 'type': 'gt', 'score': ''}
  {'video-id': '1_20', 't-start': '19.214439392089844', 't-end': '105.38209533691406', 'label': '12', 'type': 'pred', 'score': '0.3596061170101166'}
  {'video-id': '1_20', 't-start': '110.207', 't-end': '162.013', 'label': '3', 'type': 'gt', 'score': ''}
  {'video-id': '1_20', 't-start': '116.9939193725586', 't-end': '160.4468231201172', 'label': '3', 'type': 'pred', 'score': '0.13191698491573334'}
  {'video-id': '1_20', 't-start': '159.9766845703125', 't-end': '229.49961853027344', 'label': '1', 'type': 'pred', 'score': '0.1471782922744751'}

'type' value counts: {'gt': 1587, 'pred': 1559}
Max t-end across the file: 2112.1 (seconds -- consistent with a recording-length scale, not frames)


## 6. Match ActionFormer predictions to EgoVLP recordings

In [8]:
def load_pretrained_actionformer_predictions(rows, score_threshold=0.0):
    """Official pre-trained ActionFormer segments -> {rec_id: [{'segment': [s, e], 'label', 'score'}]}."""
    preds = collections.defaultdict(list)
    n_gt = 0
    for row in rows:
        if row["type"] != "pred":
            n_gt += 1
            continue
        score = float(row["score"]) if row["score"] else 0.0
        if score < score_threshold:
            continue
        rec_id = row["video-id"]
        preds[rec_id].append({
            "segment": [float(row["t-start"]), float(row["t-end"])],
            "label": row["label"],
            "score": score,
        })
    for rec_id in preds:
        preds[rec_id].sort(key=lambda d: d["segment"][0])
    return dict(preds), n_gt


actionformer_predictions, _n_gt_rows = load_pretrained_actionformer_predictions(_af_reader)
print(f"Recordings with ActionFormer predictions (released file): {len(actionformer_predictions)}")
print(f"(GT rows also present in the same file, not used here: {_n_gt_rows})")

egovlp_recording_ids = set(_egovlp_files.keys())
af_recording_ids = set(actionformer_predictions.keys())

matched_ids = sorted(egovlp_recording_ids & af_recording_ids)
skipped_no_af = sorted(egovlp_recording_ids - af_recording_ids)
af_without_egovlp = sorted(af_recording_ids - egovlp_recording_ids)

print(f"\nEgoVLP recordings available          : {len(egovlp_recording_ids)}")
print(f"ActionFormer recordings (released)   : {len(af_recording_ids)}")
print(f"Matched (have both)                  : {len(matched_ids)}")
print(f"Skipped -- EgoVLP but no AF prediction: {len(skipped_no_af)}")
print(f"AF predictions with no EgoVLP feature : {len(af_without_egovlp)} (also skipped, informational)")

Recordings with ActionFormer predictions (released file): 119
(GT rows also present in the same file, not used here: 1587)

EgoVLP recordings available          : 384
ActionFormer recordings (released)   : 119
Matched (have both)                  : 119
Skipped -- EgoVLP but no AF prediction: 265
AF predictions with no EgoVLP feature : 0 (also skipped, informational)


## 7-8. Convert timestamps, mean-pool, build per-recording records

For each matched recording, each predicted `(start, end)` in seconds is converted to feature row
indices with `int(t * FEATURE_FPS)` (identity, since `FEATURE_FPS = 1.0`), clamped to the valid
range, and mean-pooled. A segment that clamps to empty is **dropped**, not replaced with a
whole-recording fallback -- a recording with zero valid segments after that is counted as skipped,
never given an invented result.

In [9]:
def extract_step_embeddings(features, segments_sec, fps=FEATURE_FPS):
    """Mean-pool EgoVLP features inside each (start, end) segment (seconds) -> (num_steps, dim)."""
    embeddings = []
    used_segments_sec = []
    for start_sec, end_sec in segments_sec:
        start_row = max(0, int(start_sec * fps))
        end_row = min(len(features), int(end_sec * fps))
        if end_row <= start_row:
            continue
        embeddings.append(np.mean(features[start_row:end_row], axis=0))
        used_segments_sec.append((start_sec, end_sec))
    if not embeddings:
        return None, []
    return np.stack(embeddings, axis=0), used_segments_sec


def recipe_id_from_recording_id(recording_id):
    try:
        return int(recording_id.split("_")[0])
    except (ValueError, IndexError):
        return None


results = {}
n_dropped_all_segments_invalid = 0

for rec_id in matched_ids:
    features = load_egovlp_features(rec_id)
    if features is None:
        continue

    segments_sec = [tuple(p["segment"]) for p in actionformer_predictions[rec_id]]
    step_embeddings, used_segments_sec = extract_step_embeddings(features, segments_sec)

    if step_embeddings is None:
        n_dropped_all_segments_invalid += 1
        continue

    results[rec_id] = {
        "recording_id": rec_id,
        "recipe_id": recipe_id_from_recording_id(rec_id),
        "segments": used_segments_sec,   # (start, end) in seconds
        "num_steps": len(used_segments_sec),
        "step_embeddings": step_embeddings,   # (num_steps, 256)
    }

print(f"Recordings with usable step embeddings: {len(results)}")
if n_dropped_all_segments_invalid:
    print(f"Recordings dropped (had AF predictions, but none produced a valid segment): "
          f"{n_dropped_all_segments_invalid}")

Recordings with usable step embeddings: 119


## 9. Save outputs

In [10]:
import pickle
from datetime import datetime

OUTPUT_PATH = os.path.join(OUTPUT_DIR, "step_embeddings_actionformer.pkl")

output = {
    "data": results,
    "method": "actionformer_pretrained_captaincook4d",
    "source_url": PRETRAINED_AF_URL,
    "feature_fps": FEATURE_FPS,
    "feature_dim": int(_sample_feat.shape[-1]),
    "n_egovlp_recordings": len(egovlp_recording_ids),
    "n_actionformer_recordings": len(af_recording_ids),
    "n_matched": len(matched_ids),
    "n_skipped_no_actionformer": len(skipped_no_af),
    "generated_at": datetime.utcnow().isoformat() + "Z",
}

with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(output, f)

print(f"Saved {len(results)} recordings to {OUTPUT_PATH}")

Saved 119 recordings to /content/drive/MyDrive/AML_Project/extension_data/actionformer_egovlp/step_embeddings_actionformer.pkl


/tmp/ipykernel_2835/1020827974.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat() + "Z",


## 10. Verification statistics

In [11]:
print("=" * 60)
print("SUBSTEP 1 -- VERIFICATION")
print("=" * 60)
print(f"ActionFormer recordings (released predictions) : {len(af_recording_ids)}")
print(f"EgoVLP recordings available                     : {len(egovlp_recording_ids)}")
print(f"Matched (features + AF predictions, saved)      : {len(results)}")
print(f"Skipped -- EgoVLP but no AF prediction           : {len(skipped_no_af)}")
if n_dropped_all_segments_invalid:
    print(f"Skipped -- AF predictions but no valid segment   : {n_dropped_all_segments_invalid}")

steps_per_recording = [r["num_steps"] for r in results.values()]
print(f"\nSteps per recording: min={min(steps_per_recording)}, "
      f"median={sorted(steps_per_recording)[len(steps_per_recording)//2]}, "
      f"max={max(steps_per_recording)}")

_example_id = sorted(results.keys())[0]
_example = results[_example_id]
_more_suffix = " ..." if _example["num_steps"] > 3 else ""
print(f"\nExample recording: {_example_id}")
print(f"  recipe_id       : {_example['recipe_id']}")
print(f"  num_steps       : {_example['num_steps']}")
print(f"  segments (sec)  : {_example['segments'][:3]}{_more_suffix}")
print(f"  step_embeddings : shape={_example['step_embeddings'].shape}, dtype={_example['step_embeddings'].dtype}")

SUBSTEP 1 -- VERIFICATION
ActionFormer recordings (released predictions) : 119
EgoVLP recordings available                     : 384
Matched (features + AF predictions, saved)      : 119
Skipped -- EgoVLP but no AF prediction           : 265

Steps per recording: min=5, median=13, max=25

Example recording: 10_24
  recipe_id       : 10
  num_steps       : 16
  segments (sec)  : [(7.402897834777832, 52.65324020385742), (71.63810729980469, 113.92648315429688), (126.09449768066406, 167.3142852783203)] ...
  step_embeddings : shape=(16, 256), dtype=float16


---
**Context on prediction quality** (reported by the reference notebook's own analysis, not
recomputed here): measured against the GT rows in the same released file, these predictions score
tIoU>=0.5 F1 0.99, tIoU>=0.7 F1 0.62, mean best-tIoU 0.74. Segment counts run close to 1:1 with GT
steps, so the release appears pre-filtered near the GT step count -- worth keeping in mind if
Substep 2 later compares against this as if it were a from-scratch localizer's raw output.